# Multimodal Explainable AI for Early Diabetes Prediction
## Full Pipeline — Phases 1–5

| Phase | Description |
|---|---|
| 1 | Data loading, preprocessing, feature engineering |
| 2 | Synthetic clinical notes + ClinicalBERT embeddings |
| 3 | Baseline models + Early/Late Fusion + evaluation |
| 4 | SHAP explainability + LLM recommendations |
| 5 | Publication-ready figures and tables |

> **Dataset:** Diabetes 130-US Hospitals 1999–2008 (UCI, ID=296)  
> **Reproducibility:** `RANDOM_SEED = 42` throughout.

## 1. Imports & global config

All libraries for the entire pipeline imported here once.

In [2]:
import os, json, time, warnings, pickle, shutil, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from groq import Groq, RateLimitError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import shap

warnings.filterwarnings("ignore")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.facecolor":"white","axes.facecolor":"#F8F9FB",
    "axes.grid":True,"grid.alpha":0.4,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,
})
PALETTE = {
    "neg":"#4A90D9","pos":"#E05C5C","neu":"#7F77DD",
    "lr":"#4A90D9","rf":"#56B87A","xgb":"#E0AA00",
    "early":"#E05C5C","late":"#9B59B6","tab":"#7F77DD",
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("data", exist_ok=True)
os.makedirs("data/models", exist_ok=True)
os.makedirs("data/figures", exist_ok=True)
os.makedirs("data/paper", exist_ok=True)
print(f"Ready. RANDOM_SEED={RANDOM_SEED} | device={device}")

KeyboardInterrupt: 

## 2. Google Drive — mount & restore

Run at the start of **every new Colab session** to restore saved artefacts.

In [4]:
from google.colab import drive, userdata
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/diabetes_project/data"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/models", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/paper", exist_ok=True)

RESTORE_FILES = [
    "scaler.pkl","feature_names.json","diabetes_clean.parquet",
    "X_train.npy","X_val.npy","X_test.npy",
    "y_train.npy","y_val.npy","y_test.npy",
    "text_embeddings.npy","synthetic_notes_final.csv",
    "synthetic_notes.csv","notes_checkpoint.csv",
    "results_summary.csv","shap_values_xgb.npy",
    "recommendations_sample.csv",
]
for fname in RESTORE_FILES:
    src, dst = f"{DRIVE_DIR}/{fname}", f"data/{fname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"  Restored: {fname}")

for mname in ["xgboost.json","logistic_regression.pkl",
              "random_forest.pkl","early_fusion.pt","late_fusion.pt"]:
    src, dst = f"{DRIVE_DIR}/models/{mname}", f"data/models/{mname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst); print(f"Restored model: {mname}")

print("Restore complete.")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'data/scaler.pkl'

---
# Phase 2 — Synthetic Clinical Notes & ClinicalBERT Embeddings

## Rationale

Real notes unavailable (privacy). Synthetic feature-conditioned notes validated
as NLP supplement. Reference: Alshaikhdeeb et al., 2025

## 14. Groq API client

In [ ]:
client = Groq(api_key=userdata.get("GROQ_API_key"))
print("Groq client initialised.")

## 15. Sampling strategy

> `original_index` stores row position in `df_clean` (0..~100k).

In [ ]:
N_SAMPLE = 10000

df_clean_loaded = pd.read_parquet("data/diabetes_clean.parquet").reset_index(drop=True)

df_sampled = df_clean_loaded.sample(
    n=min(N_SAMPLE, len(df_clean_loaded)),
    random_state=RANDOM_SEED
).copy()

df_sampled["original_index"] = df_sampled.index
df_sampled = df_sampled.reset_index(drop=True)

print(f"Sampled: {len(df_sampled):,}")
print(f"original_index range: {df_sampled['original_index'].min()} — {df_sampled['original_index'].max()}")
print(f"Class balance: {df_sampled['target'].value_counts().to_dict()}")
print(f"Positive rate: {df_sampled['target'].mean():.1%}")

assert df_sampled["original_index"].max() > 1000, "original_index wrong!"
assert abs(df_sampled["target"].mean() - 0.112) < 0.05, "Positive rate wrong!"
print("CHECK: PASS")

## 16. Prompt engineering — token-efficient

In [ ]:
SYSTEM_PROMPT = (
    "Physician writing 2-3 sentence EHR note. "
    "Plain prose, no headers. Medical style. No names or dates."
)
AGE_DECODE = {0:"0-10",1:"10-20",2:"20-30",3:"30-40",4:"40-50",
              5:"50-60",6:"60-70",7:"70-80",8:"80-90",9:"90-100"}
A1C_DECODE = {0:"NA",1:"nl",2:">7",3:">8"}
GLU_DECODE = {0:"NA",1:"nl",2:">200",3:">300"}
MED_DECODE = {0:"no",1:"stable",2:"adjusted"}
GENDER_DECODE = {1:"M",0:"F"}

def build_user_prompt(row):
    def si(k,d=0):
        try: return int(row.get(k,d))
        except: return d
    return (
        "EHR note:\n"
        f"Pt:{GENDER_DECODE.get(si('gender'),'?')} age {AGE_DECODE.get(si('age',5),'?')}y\n"
        f"LOS:{si('time_in_hospital')}d dx:{si('number_diagnoses')} "
        f"labs:{si('num_lab_procedures')} meds:{si('num_medications')}\n"
        f"A1c:{A1C_DECODE.get(si('A1Cresult'),'NA')} "
        f"glu:{GLU_DECODE.get(si('max_glu_serum'),'NA')} "
        f"insulin:{MED_DECODE.get(si('insulin'),'no')} "
        f"metformin:{MED_DECODE.get(si('metformin'),'no')} "
        f"changed:{'yes' if si('change')==1 else 'no'}\n"
        f"prior_inpat:{si('number_inpatient')} prior_ED:{si('number_emergency')}\n"
        f"Risk:{'HIGH-readmission' if int(row['target'])==1 else 'LOW-readmission'}\n"
        "Note:"
    )
print("Prompt builder ready.")

## 17. API dry run

In [ ]:
test_msg = client.chat.completions.create(
    model="llama-3.1-8b-instant", max_tokens=80, temperature=0.4,
    messages=[{"role":"system","content":SYSTEM_PROMPT},
              {"role":"user","content":build_user_prompt(df_sampled.iloc[0])}])
print(test_msg.choices[0].message.content)
print(f"Tokens: {test_msg.usage.total_tokens}")

## 18. Batch generation — parallel with Drive checkpoint

In [ ]:
import concurrent.futures
import threading

DRIVE_CHECKPOINT = f"{DRIVE_DIR}/notes_checkpoint.csv"
DRIVE_NOTES = f"{DRIVE_DIR}/synthetic_notes.csv"
LOCAL_CHECKPOINT = "data/notes_checkpoint.csv"
NOTES_PATH = "data/synthetic_notes.csv"
CHECKPOINT_EVERY = 50
BATCH_SIZE = 3
DELAY_BETWEEN = 2.5


if os.path.exists(DRIVE_CHECKPOINT):
    shutil.copy2(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)
    print("Checkpoint restored from Drive.")
elif os.path.exists(LOCAL_CHECKPOINT):
    print("Using local checkpoint.")
else:
    print("Starting fresh.")

if os.path.exists(LOCAL_CHECKPOINT):
    checkpoint_df = pd.read_csv(LOCAL_CHECKPOINT)
    successful_df = checkpoint_df[checkpoint_df["note"] != "[GENERATION_FAILED]"]
    done_ids  = set(successful_df["original_index"].tolist())
    notes     = successful_df["note"].tolist()
    processed = successful_df["original_index"].tolist()
    targets   = successful_df["target"].tolist()
    print(f"Resuming from {len(done_ids):,} successful notes.")
    print(f"Failed previously: {len(checkpoint_df)-len(done_ids):,}")
else:
    notes, processed, targets, done_ids = [], [], [], set()

lock = threading.Lock()

def save_and_sync(processed, targets, notes, is_final=False):
    lpath = NOTES_PATH if is_final else LOCAL_CHECKPOINT
    dpath = DRIVE_NOTES if is_final else DRIVE_CHECKPOINT
    df_out = pd.DataFrame({"original_index":processed,"target":targets,"note":notes})
    if is_final:
        df_out = df_out[df_out["note"]!="[GENERATION_FAILED]"].reset_index(drop=True)
    df_out.to_csv(lpath, index=False)
    try: shutil.copy2(lpath, dpath)
    except Exception as e: print(f"  Drive sync failed: {e}")
    print(f"  Saved {'final' if is_final else 'checkpoint'}: {len(df_out)} rows")
    return df_out

def generate_single(row_tuple):
    orig_idx, target_val, row = row_tuple
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model="llama-3.1-8b-instant", max_tokens=60, temperature=0.4,
                messages=[{"role":"system","content":SYSTEM_PROMPT},
                          {"role":"user","content":build_user_prompt(row)}])
            return (orig_idx, target_val, resp.choices[0].message.content.strip())
        except RateLimitError:
            wait = 60*(attempt+1)
            tqdm.write(f"  RateLimit — waiting {wait}s...")
            time.sleep(wait)
        except Exception: time.sleep(2)
    return (orig_idx, target_val, "[GENERATION_FAILED]")

to_process = df_sampled[~df_sampled["original_index"].isin(done_ids)]
print(f"Left to generate: {len(to_process):,}")

tasks = [(int(row["original_index"]), int(row["target"]), row)
         for _, row in to_process.iterrows()]

n_completed = 0

with tqdm(total=len(tasks), desc="Generating") as pbar:
    for batch_start in range(0, len(tasks), BATCH_SIZE):
        batch = tasks[batch_start:batch_start+BATCH_SIZE]
        with concurrent.futures.ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
            results = list(executor.map(generate_single, batch))
        with lock:
            for orig_idx, target_val, note in results:
                notes.append(note); processed.append(orig_idx)
                targets.append(target_val); done_ids.add(orig_idx)
                n_completed += 1
            if n_completed % CHECKPOINT_EVERY == 0:
                save_and_sync(processed, targets, notes, is_final=False)
                failed_so_far = sum(1 for n in notes if n=="[GENERATION_FAILED]")
                tqdm.write(f"  [{n_completed}/{len(tasks)}] failed={failed_so_far}")
        pbar.update(len(batch))
        time.sleep(DELAY_BETWEEN * BATCH_SIZE)

notes_df = save_and_sync(processed, targets, notes, is_final=True)
failed   = sum(1 for n in notes if n=="[GENERATION_FAILED]")
print(f"Done. {len(notes_df):,} notes saved. Failed: {failed}")
print(f"Positive rate: {notes_df['target'].mean():.1%}  (expected ~11%)")

## 19. Build synthetic_notes_final.csv

Works from checkpoint even if NOTES_PATH doesn't exist yet.

In [ ]:
LOCAL_CHECKPOINT = "data/notes_checkpoint.csv"
DRIVE_CHECKPOINT = f"{DRIVE_DIR}/notes_checkpoint.csv"

if not os.path.exists(LOCAL_CHECKPOINT):
    if os.path.exists(DRIVE_CHECKPOINT):
        shutil.copy2(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)
        print("Checkpoint restored from Drive.")
    else:
        raise FileNotFoundError("No checkpoint found anywhere!")
else:
    print("Using local checkpoint.")

checkpoint_df = pd.read_csv(LOCAL_CHECKPOINT)
notes_df = checkpoint_df[checkpoint_df["note"]!="[GENERATION_FAILED]"].reset_index(drop=True)
notes_df["embedding_idx"] = range(len(notes_df))
notes_df.to_csv("data/synthetic_notes_final.csv", index=False)
try: shutil.copy2("data/synthetic_notes_final.csv", f"{DRIVE_DIR}/synthetic_notes_final.csv")
except: pass

print(f"Checkpoint total  : {len(checkpoint_df):,}")
print(f"Failed (excluded) : {(checkpoint_df['note']=='[GENERATION_FAILED]').sum():,}")
print(f"Successful (used) : {len(notes_df):,}")
print(f"Positive rate     : {notes_df['target'].mean():.1%}")
print(f"original_index    : {notes_df['original_index'].min()} — {notes_df['original_index'].max()}")
assert notes_df["original_index"].max() > 1000, "original_index wrong!"
print("CHECK: PASS")

## 20. Quality validation

In [ ]:
notes_df["word_count"] = notes_df["note"].str.split().str.len()
notes_df["has_any_keyword"] = notes_df["note"].apply(
    lambda t: bool(re.search(
        r"\b(glucose|insulin|diabetes|glycemic|HbA1c|A1c|hyperglycemia|metformin)\b",t,re.I)))
hit_rate = notes_df["has_any_keyword"].mean()
N_DIV    = min(300,len(notes_df))
sample_t = notes_df["note"].sample(N_DIV,random_state=RANDOM_SEED).tolist()
tfidf_m  = TfidfVectorizer(max_features=3000,stop_words="english").fit_transform(sample_t)
sim_vals = cosine_similarity(tfidf_m)[np.triu_indices(N_DIV,k=1)]
mean_sim = sim_vals.mean()

print(f"Mean words : {notes_df['word_count'].mean():.1f}  ({'PASS' if notes_df['word_count'].mean()>=20 else 'FAIL'})")
print(f"Keyword hit: {hit_rate:.1%}  ({'PASS' if hit_rate>=0.8 else 'FAIL'})")
print(f"TF-IDF sim : {mean_sim:.4f}  ({'PASS' if mean_sim<0.5 else 'WARN'})")

## 21. Bio+ClinicalBERT embeddings

Cached — skips computation if file exists with correct size.

In [ ]:
LOCAL_EMB = "data/text_embeddings.npy"
DRIVE_EMB = f"{DRIVE_DIR}/text_embeddings.npy"

loaded = False
for src in [LOCAL_EMB, DRIVE_EMB]:
    if os.path.exists(src):
        tmp = np.load(src)
        if tmp.shape[0] == len(notes_df):
            if src == DRIVE_EMB: shutil.copy2(src, LOCAL_EMB)
            text_embeddings = tmp
            print(f"Loaded from cache: {text_embeddings.shape}")
            loaded = True
            break
        else:
            print(f"Cache size mismatch ({tmp.shape[0]} vs {len(notes_df)}) — recomputing")
            os.remove(src)

if not loaded:
    MODEL_NAME     = "emilyalsentzer/Bio_ClinicalBERT"
    print(f"Loading {MODEL_NAME} ...")
    bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    bert_model_emb = AutoModel.from_pretrained(MODEL_NAME).to(device)
    bert_model_emb.eval()

    def encode_notes_batch(texts, batch_size=32):
        all_vecs = []
        for start in tqdm(range(0,len(texts),batch_size),desc="Encoding"):
            batch   = texts[start:start+batch_size]
            encoded = bert_tokenizer(batch,padding=True,truncation=True,
                                     max_length=128,return_tensors="pt")
            encoded = {k:v.to(device) for k,v in encoded.items()}
            with torch.no_grad():
                out = bert_model_emb(**encoded)
            all_vecs.append(out.last_hidden_state[:,0,:].cpu().numpy())
        vecs = np.vstack(all_vecs)
        return vecs / np.linalg.norm(vecs,axis=1,keepdims=True)

    text_embeddings = encode_notes_batch(notes_df["note"].tolist())
    np.save(LOCAL_EMB, text_embeddings.astype(np.float32))
    shutil.copy2(LOCAL_EMB, DRIVE_EMB)
    print(f"Computed and saved: {text_embeddings.shape}")

print(f"Shape : {text_embeddings.shape}")
print(f"Norm : {np.linalg.norm(text_embeddings[0]):.6f}  (should be ~1.0)")
assert text_embeddings.shape[0]==len(notes_df), "Shape mismatch!"
print("CHECK: PASS")